In [1]:
import pandas as pd

# Load the training and test datasets
train_data = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/04_diabetes/train.csv')
test_data = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/04_diabetes/test.csv')

# Drop duplicate rows from both datasets
train_data_cleaned = train_data.drop_duplicates()
test_data_cleaned = test_data.drop_duplicates()

# Verify the number of rows before and after dropping duplicates
print(f"Train data rows before: {len(train_data)}, after: {len(train_data_cleaned)}")
print(f"Test data rows before: {len(test_data)}, after: {len(test_data_cleaned)}")


Train data rows before: 80000, after: 77430
Test data rows before: 20000, after: 19807


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Using the cleaned train data to get the latest column information
column_info = get_column_info(train_data_cleaned)
print("column_info")
print(column_info)


2025-08-30 18:34:22.271 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['gender', 'smoking_history'], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import OneHotEncode

# Initialize the OneHotEncode tool for the specified columns
one_hot_encoder = OneHotEncode(features=['gender', 'smoking_history'])

# Fit the encoder on the training data and transform both training and test data
train_data_encoded = one_hot_encoder.fit_transform(train_data_cleaned.copy())
test_data_encoded = one_hot_encoder.transform(test_data_cleaned.copy())

# Display the first few rows of the encoded datasets to verify the changes
train_data_encoded.head(), test_data_encoded.head()


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


(    age  hypertension  ...  smoking_history_never  smoking_history_not current
 0  73.0             0  ...                    0.0                          0.0
 1  80.0             0  ...                    1.0                          0.0
 2  38.0             0  ...                    0.0                          0.0
 3  26.0             0  ...                    0.0                          1.0
 4  61.0             1  ...                    0.0                          0.0
 
 [5 rows x 16 columns],
     age  hypertension  ...  smoking_history_never  smoking_history_not current
 0  13.0             0  ...                    0.0                          0.0
 1   3.0             0  ...                    0.0                          0.0
 2  63.0             0  ...                    0.0                          0.0
 3   2.0             0  ...                    1.0                          0.0
 4  33.0             0  ...                    0.0                          1.0
 
 [5 rows x 1

In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Using the latest DataFrame from the finished tasks
column_info_train = get_column_info(train_data_encoded)
column_info_test = get_column_info(test_data_encoded)

print("Train data column_info")
print(column_info_train)
print("\nTest data column_info")
print(column_info_test)


Train data column_info
{'Category': [], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'], 'Datetime': [], 'Others': []}

Test data column_info
{'Category': [], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'], 'Datetime': [], 'Others': []}


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
import numpy as np

# Copy the data to avoid modifying the original
train_data_encoded_copy = train_data_encoded.copy()
test_data_encoded_copy = test_data_encoded.copy()

# Define the label column
label_col = 'diabetes'

# Separate features and target
X_train = train_data_encoded_copy.drop(columns=[label_col])
y_train = train_data_encoded_copy[label_col]

X_test = test_data_encoded_copy.drop(columns=[label_col])
y_test = test_data_encoded_copy[label_col]

# Build a random forest classifier to rank the most important features
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Get the feature importances
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

# Print the feature ranking
print("Feature ranking:")
for f in range(X_train.shape[1]):
    print(f"{f + 1}. {X_train.columns[indices[f]]} ({importances[indices[f]]:.3f})")

# Select the most important features
sfm = SelectFromModel(rf, threshold=0.05)
X_train_important = sfm.transform(X_train)
X_test_important = sfm.transform(X_test)

# Print the number of selected features
print(f"Number of selected features: {X_train_important.shape[1]}")


Feature ranking:
1. HbA1c_level (0.408)
2. blood_glucose_level (0.312)
3. bmi (0.126)
4. age (0.105)
5. hypertension (0.015)
6. heart_disease (0.011)
7. smoking_history_No Info (0.004)
8. smoking_history_former (0.004)
9. smoking_history_never (0.003)
10. smoking_history_current (0.003)
11. gender_Male (0.002)
12. gender_Female (0.002)
13. smoking_history_not current (0.002)
14. smoking_history_ever (0.002)
15. gender_Other (0.000)
Number of selected features: 4


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
